![](https://i.imgur.com/N0zUCi0.png)

# 🔬 Mini Project 1 - Assignment: Build a Pipeline for Multimodal Research and Report Generation


## 📋 Project Overview

In this project, you will demonstrate advanced multimodal AI capabilities by building an end-to-end research pipeline using **Google Gemini**. You'll ingest diverse data formats—PDFs, images, videos, and audio—and automatically generate comprehensive, academically-structured research reports with proper citations.

You will explore **two distinct architectural approaches** for multimodal analysis, comparing their strengths, tradeoffs, and use cases in real-world applications.

---

## 🎯 Objectives

- **Multimodal Data Integration**: You will process and analyze text documents, architectural diagrams, and video presentations simultaneously
- **Cross-Modal Reasoning**: You will enable the AI to identify relationships and insights across different data formats
- **Automated Report Generation**: You will produce professional research reports (1500-2000 words) with academic citations
- **Architectural Comparison**: You will evaluate single-pass vs. two-phase processing approaches

---

## 🏗️ Architecture Approaches

### Approach 1: Single-Step Multimodal Analysis
**Strategy**: You will encode and process all resources in one unified API call
**Advantage**: Holistic cross-modal reasoning, single API call efficiency
**Best For**: When cross-modal relationships are critical and all resources fit within context window

### Approach 2: Two-Phase Sequential Analysis
**Strategy**: You will individually analyze each resource, aggregate the text, and perform a final synthesis
**Advantage**: Modality-specific optimization, incremental processing, token efficiency
**Best For**: Large files, when specialized analysis per modality is needed, or for modular workflows

---

## 🛠️ Technical Stack

- **LLM**: Google Gemini 3.0 Flash (multimodal capabilities)
- **Framework**: LangChain (message handling and prompt orchestration)
- **Data Processing**: Python (base64 encoding, MIME type detection)

---

## 📊 Pipeline Workflow

You will build a pipeline that moves through the following stages:

1. **Resource Preparation**: You will implement automatic MIME type detection and base64 encoding
2. **Prompt Engineering**: You will craft structured instructions for academic report generation
3. **Multimodal Processing**: You will have the AI analyze text, visuals, and video content
4. **Citation Integration**: You will implement automatic source attribution with numbered references
5. **Report Synthesis**: You will generate a cohesive, professionally-formatted output

---

## 💡 Key Features

✅ **Automatic File Processing**: You will build a utility function that handles any file type

✅ **Academic Citations**: You will implement Wikipedia/research paper style references [1][2][3]

✅ **Unified Narrative**: You will instruct the AI to avoid explicit source-type mentions in the final report

✅ **Conflict Resolution**: You will see how the AI reconciles contradictions across sources

✅ **Comparative Analysis**: You will conduct a side-by-side evaluation of both approaches

---

## 🎓 Learning Outcomes

By completing this project, you will understand:
- How to build production-ready multimodal AI pipelines
- The tradeoffs between different architectural patterns for LLM applications
- Best practices for prompt engineering in multimodal contexts
- Techniques for generating structured, citation-backed AI outputs

---

**Happy Building!** 🚀

### Installation

We start by installing all necessary libraries.
These include LangChain, Google Generative AI SDKs, and supporting utilities.

In [ ]:
!pip install -q langchain==1.2.8
!pip install -q langchain-community==0.4.1
!pip install -q langchain-google-genai==4.2.0

### Import Dependencies

In [ ]:
# Required imports
import os
import base64
from getpass import getpass
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.messages import HumanMessage
from IPython.display import Markdown

### Securely Set the Google API Key

For security,  avoid hardcoding the API key.
If the key is not already set, the user is prompted to enter it securely.

In [ ]:
# Set API Key securely
os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google API key: ")

## Download Research Resources

You will use `gdown` to pull the research resource data into your local environment.

- Two PNG images
- One PDF specification document
- One explanatory video

You will later upload these assets to Gemini for multimodal reasoning.

The overall idea is that you will do some general research, find relevant resources, download them, and leverage LLMs to help understand these resources and build a report.

In [ ]:
!gdown 1AeWLyY3qHVkWyx9LAD3FYGfhcc9dB96c

In [ ]:
!unzip -o AIAP_proj1_resources.zip

### Preview the Multimodal Data

Before analysis, you will preview the data manually by downloading files from the Colab workspace, as depicted in the image below.

![](https://i.imgur.com/FsFHGZE.png)

You can also open the files directly in the notebook, as shown below.

In [ ]:
from IPython.display import Image as ImageDisp, display

display(ImageDisp('AIAP_proj1_resources/Context_window_image_1.png'))

In [ ]:
display(ImageDisp("AIAP_proj1_resources/Context_window_image_2.png"))

In [ ]:
from IPython.display import Markdown, Video

Video('AIAP_proj1_resources/Context_window_video.mp4', embed=True, width=450)

In [ ]:
import IPython

IPython.display.Audio('AIAP_proj1_resources/Context_window_vs_Rag_audio.mp3')

## Create a File Processing Utility for Multimodal Prompting

### Overview

You will create a utility function that simplifies the process of preparing local files for use with Multimodal LLMs like Google's Generative AI models (like Gemini). It will automatically detect the file type and convert it to the base64-encoded format required by the LangChain Google GenAI integration.

#### Function: `process_local_file(file_path: str) -> dict`

#### Purpose
You will build this function to convert any local file (images, videos, audio, PDFs, etc.) into the standardized format needed for `HumanMessage` content blocks when working with `ChatGoogleGenerativeAI`.

#### Your function will:
1. **Validate** that the file exists at the given path
2. **Detect** the MIME type automatically using Python's `mimetypes` module
3. **Read and encode** the file content as base64
4. **Categorize** the content type as:
   - `"image"` for image files (JPEG, PNG, GIF, etc.)
   - `"video"` for video files (MP4, AVI, MOV, etc.)
   - `"audio"` for audio files (MP3, WAV, etc.)
   - `"file"` for documents (PDF, TXT, etc.)

### Returns
Your function will return a dictionary ready to be inserted into a `HumanMessage` content list:
```python
{
    "type": "image" | "video" | "audio" | "file",
    "base64": "<base64-encoded-content>",
    "mime_type": "image/jpeg" | "video/mp4" | "audio/mpeg" | etc.
}
```

### Usage Example

Once built, you will use your function like this:
```python
from langchain.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI

# Initialize model
model = ChatGoogleGenerativeAI(model="gemini-3-pro-preview")

# Process your local file
file_content = process_local_file("path/to/your/image.jpg")

# Create message with the processed file
message = HumanMessage(
    content=[
        {"type": "text", "text": "What's in this image?"},
        file_content  # Insert the processed file directly
    ]
)

# Get response
response = model.invoke([message])
```

### Error Handling
Your function should raise:
- `FileNotFoundError` if the file doesn't exist
- `ValueError` if the MIME type cannot be determined

**YOUR TURN: Implement this function below** ↓

## Initialize the Gemini Model


In [ ]:
llm = ChatGoogleGenerativeAI(
    # here you can go with gemini 3 Flash which is their latest LLM
    # it is also faster than pro however less powerful
    model="gemini-3-flash-preview",
    temperature=0
)


## Approach 1: Single-Step Multimodal Analysis and Report Generation

### Overview

In this approach, you will leverage the native multimodal capabilities of Google's Gemini models to process and analyze multiple data formats simultaneously in a single API call. You will encode all source materials (text, images, PDFs, audio and video) into a unified prompt, allowing the model to perform cross-modal reasoning and generate a cohesive research report that synthesizes insights from all modalities at once.

![](https://i.imgur.com/Itezl0t.png)


---

### Methodology

#### Step 1: Multimodal Resource Preparation
You will process all research materials using the `process_local_file()` utility function you built earlier. For each file, the function will:
- Automatically detect its MIME type
- Read and encode its contents as base64
- Categorize its content type (image, video, file, audio)
- Return a standardized dictionary ready for LLM consumption

#### Step 2: Unified Prompt Construction
You will construct a single `HumanMessage` object containing:
- **System Instructions**: Detailed guidelines for report structure, citation style, and academic tone
- **Multimodal Resources**: All encoded materials (images, PDFs, videos) embedded directly in the content array
- **Research Directives**: Specific sections to address and the analytical framework to follow

#### Step 3: Single-Pass Generation
You will pass all inputs to the model in one call, enabling it to:
- Reference relationships between visual diagrams, textual documentation, and video demonstrations
- Integrate information from different sources naturally without segmentation
- Support claims with numbered references to specific source materials

### Expected Output

Your pipeline should produce a comprehensive research report (1500-2000 words) containing:
- Executive summary and introduction
- Technical architecture and operational workflow analysis
- Ecosystem integration and capabilities assessment
- Critical evaluation of strengths and limitations
- Academic-style numbered citations [1], [2], [3], [4]
- Professional, unified narrative voice

---

**YOUR TURN: Implement this approach below** ↓



Display the final report here in Markdown

## Approach 2: Two-Phase Multimodal Analysis and Report Generation

### Overview

In this approach, you will employ a decomposed, sequential methodology where you analyze each resource type independently before synthesizing everything into a unified report. Unlike Approach 1 where you processed everything in a single pass, here you will first extract modality-specific insights from each resource, then aggregate them through a text-based synthesis step. This separation will allow you to apply targeted analysis strategies tailored to each resource type while still producing a cohesive final output.

![](https://i.imgur.com/QScQoQN.png)

---

### Methodology

#### Phase 1: Individual Resource Analysis

**Step 1.1: Technical Document Analysis (PDF)**
- You will process the PDF document using `process_local_file()`
- Extract key concepts, specifications, goals, and design principles
- Generate a resource description for citation purposes
- Store your analysis in a `pdf_analysis` variable

**Step 1.2: Visual Architecture Analysis (Images)**
- You will process the architecture diagrams using `process_local_file()`
- Analyze system components, data flows, and architectural patterns
- Generate a resource description identifying what the diagrams illustrate
- Store your analysis in an `images_analysis` variable

**Step 1.3: Presentation Analysis (Video)**
- You will process the video presentation using `process_local_file()`
- Extract demonstrations, use cases, and verbal explanations
- Generate a resource description summarizing the presentation content
- Store your analysis in a `video_analysis` variable

**Step 1.4: Audio Analysis**
- You will process the audio file using `process_local_file()`
- Extract spoken content, key points, and any additional context it provides
- Generate a resource description summarizing the audio content
- Store your analysis in an `audio_analysis` variable

#### Phase 2: Aggregated Report Synthesis

You will then pass all four individual analyses as text inputs into a final synthesis step where you will:
- **Combine Insights**: Merge information from all four analyses into a single context
- **Apply a Unified Prompt**: Use the same comprehensive report structure as Approach 1
- **Generate Citations**: Map claims to source materials [1] PDF, [2] Images, [3] Video, [4] Audio
- **Produce a Cohesive Output**: Create a seamless narrative that integrates all insights

### Expected Output

#### Phase 1 Outputs:
You will produce four individual analyses, each containing:
- A 1-2 sentence resource description (for citation tracking)
- Modality-specific insights (textual concepts, visual patterns, demonstrated features, or spoken content)
- Structured prose ready for aggregation

#### Phase 2 Output:
Your pipeline should produce a comprehensive research report (1500-2000 words) containing:
- Executive summary and introduction
- Technical architecture and operational workflow analysis
- Ecosystem integration and capabilities assessment
- Critical evaluation of strengths and limitations
- Academic-style numbered citations [1], [2], [3], [4]
- Professional, unified narrative voice synthesized from all analyses

---

**YOUR TURN: Implement this approach below** ↓

### Phase 1: Individual Resource Analysis

Extract insights from each resource type separately with targeted prompts.
Each analysis includes a brief description of the resource for later citation purposes.

#### Step 1.1: Analyze Technical Document (PDF)


#### Step 1.2: Analyze Architecture Diagrams (Images)

#### Step 1.3: Analyze Presentation Video


#### Step 1.4: Analyze Audio Data


### Phase 2: Aggregated Report Synthesis

Combine all individual analyses into a comprehensive, unified research report.


Display the final report here in Markdown